In [2]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Project root from notebook location
PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

# 1. Load historical data
LOCAL = PROJECT_ROOT / "data" / "raw" / "player_history_last_season.csv"
URL = (
    "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/"
    "master/data/2024-25/gws/merged_gw.csv"
)

if LOCAL.exists():
    hist = pd.read_csv(LOCAL)
    print("Loaded from cache")
else:
    hist = pd.read_csv(URL)
    LOCAL.parent.mkdir(parents=True, exist_ok=True)
    hist.to_csv(LOCAL, index=False)
    print(f"Downloaded {len(hist):,} rows")

# 2. Select columns (fixed typo: "assists")
columns = [
    "element",
    "player_name",
    "round",
    "total_points",
    "minutes",
    "goals_scored",
    "assists",
    "ict_index",
    "influence",
    "creativity",
    "threat",
    "selected_by_percent",
    "was_home",
    "opponent_team",
    "kickoff_time",
]
existing_columns = [col for col in columns if col in hist.columns]
hist = hist[existing_columns]

# 3. Numeric conversion
numeric_cols = [
    "round",
    "total_points",
    "minutes",
    "goals_scored",
    "assists",
    "ict_index",
    "influence",
    "creativity",
    "threat",
    "selected_by_percent",
]
for col in numeric_cols:
    if col in hist.columns:
        hist[col] = pd.to_numeric(hist[col], errors="coerce")

hist = hist.sort_values(["element", "round"])

# 4. Lagged rolling features (no leakage)
hist["minutes_l3"] = hist.groupby("element")["minutes"].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
hist["points_l3"] = hist.groupby("element")["total_points"].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
hist["ict_l3"] = hist.groupby("element")["ict_index"].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
hist["minutes_l5"] = hist.groupby("element")["minutes"].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
)
hist["points_l5"] = hist.groupby("element")["total_points"].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
)

hist["target"] = hist.groupby("element")["total_points"].shift(-1)

# 5. Time-based split (no leakage!)
train = hist[hist["round"] <= 30].dropna(subset=["target"])
test = hist[hist["round"] > 30].dropna(subset=["target"])

feature_columns = [
    "minutes_l3",
    "points_l3",
    "ict_l3",
    "minutes_l5",
    "points_l5",
    "minutes",
    "ict_index",
    "selected_by_percent",
    "was_home",
]
existing_features = [col for col in feature_columns if col in train.columns]

X_train, y_train = train[existing_features], train["target"]
X_test, y_test = test[existing_features], test["target"]

# 6. Train model
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

# 7. Compute all metrics
mae = mean_absolute_error(y_test, preds)
rmse = float(np.sqrt(mean_squared_error(y_test, preds)))  # FIXED: sqrt
spearman = pd.Series(preds).corr(pd.Series(y_test.to_numpy()), method="spearman")
baseline_preds = test["points_l3"].fillna(0)
baseline_mae = mean_absolute_error(y_test, baseline_preds)
# Captain hit rate
test_copy = test.copy()
test_copy["pred"] = preds
test_copy["gw"] = test_copy["round"] + 1
hits = total = 0
for gw, grp in test_copy.groupby("gw"):
    if len(grp) < 10:
        continue
    best = grp.loc[grp["target"].idxmax(), "element"]
    if best in grp.nlargest(10, "pred")["element"].values:
        hits += 1
    total += 1
captain_hit_rate = hits / total if total else 0.0

# 8. Save everything
metrics = {
    "season": "2024-25",
    "test_gameweeks": "31-38",
    "n_test_rows": int(len(y_test)),
    "mae": round(float(mae), 3),
    "rmse": round(float(rmse), 3),
    "spearman": round(float(spearman), 3),
    "baseline_mae": round(float(baseline_mae), 3),
    "captain_hit_rate": round(float(captain_hit_rate), 3),
}
(MODEL_DIR / "model_metrics.json").write_text(json.dumps(metrics, indent=2))

pd.DataFrame(
    {"feature": existing_features, "importance": model.feature_importances_}
).sort_values("importance", ascending=False).to_csv(
    MODEL_DIR / "feature_importance.csv", index=False
)

joblib.dump(model, MODEL_DIR / "fpl_points_model.joblib")
joblib.dump(existing_features, MODEL_DIR / "feature_columns.joblib")

print(json.dumps(metrics, indent=2))
print(f"Saved to: {MODEL_DIR}")

Loaded from cache
{
  "season": "2024-25",
  "test_gameweeks": "31-38",
  "n_test_rows": 5663,
  "mae": 1.091,
  "rmse": 2.092,
  "spearman": 0.68,
  "baseline_mae": 1.099,
  "captain_hit_rate": 0.143
}
Saved to: d:\fpl-edge\models


In [5]:
import pandas as pd

# 1. Define the URL for the 2024-25 season's merged gameweek data
url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2024-25/gws/merged_gw.csv"

# 2. Read the CSV directly into a pandas DataFrame
print("Downloading historical data from GitHub...")
hist = pd.read_csv(url)

# 3. Save it locally in your raw folder so you don't have to re-download it every time
hist.to_csv("../scripts/data/raw/player_history_last_season.csv", index=False)

# 4. Show the results
print(f"✅ Successfully loaded {len(hist):,} rows of historical data!")
hist.head()

✅ Successfully loaded 27,605 rows of historical data!


,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,...,1,0.0,2,0,0,0,50,False,0,1
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.2,0,0,0,0,0.0,427,...,1,0.0,0,0,0,0,45,True,0,1
2,Tomiyasu Takehiro,DEF,Arsenal,0.0,0,0,0,0,0.0,22,...,2,0.0,0,0,0,0,50,True,0,1
3,Malcolm Ebiowei,MID,Crystal Palace,0.0,0,0,0,0,0.0,197,...,2,0.0,0,0,0,0,45,False,0,1
4,Ben Brereton Díaz,MID,Southampton,1.0,0,0,-2,0,14.0,584,...,1,16.0,1,0,0,0,55,False,1,1


In [6]:
hist.columns

Index(['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps',
       'clean_sheets', 'creativity', 'element', 'expected_assists',
       'expected_goal_involvements', 'expected_goals',
       'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored',
       'ict_index', 'influence', 'kickoff_time', 'minutes', 'mng_clean_sheets',
       'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw',
       'mng_underdog_win', 'mng_win', 'modified', 'opponent_team', 'own_goals',
       'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves',
       'selected', 'starts', 'team_a_score', 'team_h_score', 'threat',
       'total_points', 'transfers_balance', 'transfers_in', 'transfers_out',
       'value', 'was_home', 'yellow_cards', 'GW'],
      dtype='str')

In [7]:
columns = [
    "element","player_name","round","total_points","minutes",
    "goals_scored","assits","ict_index","influence","creativity",
    "threat","selected_by_percent","was_home","opponent_team","kickoff_time"
]

existing_columns = [col for col in columns if col in hist.columns]
hist = hist[existing_columns]
hist.head()

,element,round,total_points,minutes,goals_scored,ict_index,influence,creativity,threat,was_home,opponent_team,kickoff_time
0,77,1,2,62,0,3.6,22.8,12.8,0.0,False,16,2024-08-17T14:00:00Z
1,427,1,0,0,0,0.0,0.0,0.0,0.0,True,3,2024-08-17T14:00:00Z
2,22,1,0,0,0,0.0,0.0,0.0,0.0,True,20,2024-08-17T14:00:00Z
3,197,1,0,0,0,0.0,0.0,0.0,0.0,False,4,2024-08-18T13:00:00Z
4,584,1,1,70,0,3.3,2.6,14.0,16.0,False,15,2024-08-17T14:00:00Z


In [8]:
numeric_cols = [
    "round",
    "total_points",
    "minutes",
    "goals_scored",
    "assists",
    "ict_index",
    "influence",
    "creativity",
    "threat",
    "selected_by_percent",
]

for col in numeric_cols:
    if col in hist.columns:
        hist[col] = pd.to_numeric(hist[col], errors="coerce")

In [9]:
hist = hist.sort_values(["element","round"])

In [10]:
hist["minutes_l3"] = (
    hist.groupby("element")["minutes"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
hist["points_l3"] = (
    hist.groupby("element")["total_points"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
hist["ict_l3"] = (
    hist.groupby("element")["ict_index"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
hist["minutes_l5"] = (
    hist.groupby("element")["minutes"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)
hist["points_l5"] = (
    hist.groupby("element")["total_points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)



In [11]:
hist["target"] = hist.groupby("element")["total_points"].shift(-1)

In [12]:
train = hist.dropna(subset=["target"]).copy()

In [13]:
feature_columns = [
    "minutes_l3",
    "points_l3",
    "ict_l3",
    "minutes_l5",
    "points_l5",
    "minutes",
    "ict_index",
    "selected_by_percent",
    "was_home",
]

existing_features = [col for col in feature_columns if col in train.columns]

X = train[existing_features]
y = train["target"]

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")


MAE: 1.142
RMSE: 4.432


In [15]:
import pandas as pd

importance = pd.DataFrame({
    "feature":existing_features,
    "importance": model.feature_importances_
}).sort_values("importance",ascending=False)

importance

,feature,importance
5,minutes,0.263522
4,points_l5,0.158150
2,ict_l3,0.142298
6,ict_index,0.125895
3,minutes_l5,0.107261
1,points_l3,0.096294
0,minutes_l3,0.081704
7,was_home,0.024877


In [16]:
import joblib 
from pathlib import Path

Path("models").mkdir(exist_ok=True)

joblib.dump(model, "models/fpl_points_model.joblib")

['models/fpl_points_model.joblib']